In [72]:
import pandas as pd
faq=pd.read_csv("../data/faq.csv")
products=pd.read_csv("../data/products.csv")
print('faq',faq.shape)
print('products',products.shape)

faq (10, 2)
products (20, 10)


In [73]:
faq_documents = []

for _, row in faq.iterrows():
    document = f"""
Question: {row['question']}

Answer: {row['answer']}
"""
    faq_documents.append(document.strip())

In [74]:
product_documents = []

for _, row in products.iterrows():
    document = f"""
Product: {row['name']}

Category: {row['category']}

Description: {row['description']}

Price: ₹{row['price']}

Color: {row['color']}

Available sizes: {row['size'].replace('|', ', ')}

Material: {row['material']}

Occasion: {row['occasion'].replace('|', ', ')}

Style: {row['style']}
"""
    product_documents.append(document.strip())

In [75]:
from sentence_transformers import SentenceTransformer 
model=SentenceTransformer("all-MiniLM-L6-v2")
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [76]:
text='what is your return policy'
embedding=model.encode(text)
print(embedding)

[-6.11513061e-03  4.12707701e-02  5.52058481e-02 -1.02123329e-02
  2.48977300e-02  3.57400998e-02  5.64216450e-02 -4.82230745e-02
 -6.03492893e-02  1.29634747e-02  6.72657043e-02  2.23971922e-02
 -3.96994278e-02 -4.07650508e-02 -2.45336406e-02 -6.84003308e-02
  2.52037942e-02  2.50932537e-02 -8.01257864e-02  2.80040875e-03
  2.47590654e-02 -3.80776487e-02 -9.43763647e-03  1.67666990e-02
  3.19933444e-02  3.11157592e-02  3.55342217e-03  4.10565957e-02
 -9.57018584e-02 -4.75557037e-02  5.14999777e-02 -4.02723849e-02
 -8.64089131e-02 -5.23020029e-02 -5.48457690e-02  8.33269507e-02
 -6.60834685e-02 -1.40192255e-01 -5.84819838e-02  2.98267081e-02
 -4.00313400e-02  2.82919183e-02  1.06480019e-02  2.87542716e-02
  9.34704468e-02 -1.53062055e-02  6.73836097e-02  3.52033861e-02
  6.59756586e-02  3.50789763e-02  9.10791606e-02  7.00869560e-02
 -2.19531283e-02  2.78750882e-02  2.23886017e-02  5.54581583e-02
  2.11470425e-02 -2.47397944e-02 -3.40169407e-02 -1.98672246e-02
  7.11535141e-02 -9.88058

In [77]:
print(embedding.shape)

(384,)


In [78]:
faq_embeddings=model.encode(faq_documents)
print(faq_embeddings.shape)

(10, 384)


In [79]:
product_embeddings=model.encode(product_documents)
print(product_embeddings.shape)

(20, 384)


In [80]:
query='Can i get my money back?'
query_embedding=model.encode(query)
print(query_embedding.shape)

(384,)


In [81]:
from sklearn.metrics.pairwise import cosine_similarity
similarities=cosine_similarity(
    [query_embedding],
    faq_embeddings
)[0]
print(similarities)

[ 0.3292696   0.26695    -0.04410848  0.08764052  0.27548838  0.14535898
  0.13988815  0.3826117   0.28691947  0.1293602 ]


In [82]:
best_index=similarities.argmax()
print('best matching faq index:',best_index)

best matching faq index: 7


In [83]:
print(faq_documents[best_index])

Question: Can I return a sale item?

Answer: Sale items can be returned only if they meet the applicable return conditions.


In [84]:
def retrieve_faq(query, top_k=3):
    query_embedding = model.encode(query)

    similarities = cosine_similarity(
        [query_embedding],
        faq_embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "document": faq_documents[index],
            "score": similarities[index]
        })

    return results

In [85]:
results=retrieve_faq('can i exchange a product?')

In [86]:
for result in results:
    print('similarity',result['score'])
    print(result['document'])
    print("-"*50)

similarity 0.7728349
Question: Can I exchange a product?

Answer: Yes, eligible products can be exchanged for another size within 30 days of delivery.
--------------------------------------------------
similarity 0.47671634
Question: Can I return a sale item?

Answer: Sale items can be returned only if they meet the applicable return conditions.
--------------------------------------------------
similarity 0.4548831
Question: What is your return policy?

Answer: You can return eligible products within 30 days of delivery. Products must be unused and in their original condition with tags attached.
--------------------------------------------------


In [87]:
def retrieve_products(query, top_k=3):
    query_embedding = model.encode(query)

    similarities = cosine_similarity(
        [query_embedding],
        product_embeddings
    )[0]

    top_indices = similarities.argsort()[::-1][:top_k]

    results = []

    for index in top_indices:
        results.append({
            "document": product_documents[index],
            "score": similarities[index],
            "index": index
        })

    return results

In [88]:
results = retrieve_products("I want a casual outfit for college")

for result in results:
    index = result["index"]

    print("Product ID:", products.iloc[index]["product_id"])
    print("Product:", products.iloc[index]["name"])
    print("Similarity:", result["score"])
    print("-" * 50)

Product ID: P009
Product: Black Straight Fit Trousers
Similarity: 0.52527905
--------------------------------------------------
Product ID: P014
Product: Navy Blue Polo T-Shirt
Similarity: 0.49951395
--------------------------------------------------
Product ID: P020
Product: Lavender Casual Skirt
Similarity: 0.4942707
--------------------------------------------------


In [89]:
filtered_products=products[products['price']<=1500]
print(filtered_products[['product_id','name','price']])

   product_id                        name  price
0        P001  Classic White Cotton Shirt   1299
5        P006      Black Slim Fit T-Shirt    799
9        P010             Pink Ribbed Top    999
10       P011       White Casual Sneakers   1499
13       P014      Navy Blue Polo T-Shirt   1199
17       P018     Green Oversized T-Shirt    899
19       P020       Lavender Casual Skirt   1499


In [90]:
def filter_products(
    max_price=None,
    color=None,
    occasion=None,
    style=None,
    category=None
):

    filtered = products.copy()

    # Price filter
    if max_price is not None:
        filtered = filtered[
            filtered["price"] <= max_price
        ]

    # Color filter
    if color is not None:
        filtered = filtered[
            filtered["color"].str.lower() == color.lower()
        ]

    # Occasion filter
    if occasion is not None:
        filtered = filtered[
            filtered["occasion"].str.lower().str.contains(
                occasion.lower(),
                na=False
            )
        ]

    # Style filter
    if style is not None:
        filtered = filtered[
            filtered["style"].str.lower() == style.lower()
        ]

    # Category filter
    if category is not None:
        filtered = filtered[
            filtered["category"].str.lower() == category.lower()
        ]

    return filtered

In [91]:
def retrieve_filtered_products(
    query,
    max_price=None,
    color=None,
    occasion=None,
    style=None,
    category=None,
    top_k=3
):

    filtered = filter_products(
        max_price=max_price,
        color=color,
        occasion=occasion,
        style=style,
        category=category
    )

    if filtered.empty:
        return []

    # Get the original dataframe indices
    filtered_indices = filtered.index.tolist()

    # Create embedding for the user's query
    query_embedding = model.encode(query)

    # Compare query with only the filtered products
    similarities = cosine_similarity(
        [query_embedding],
        product_embeddings[filtered_indices]
    )[0]

    # Get the most relevant products
    top_positions = similarities.argsort()[::-1][:top_k]

    results = []

    for position in top_positions:

        original_index = filtered_indices[position]

        results.append({
            "product_id": products.iloc[original_index]["product_id"],
            "name": products.iloc[original_index]["name"],
            "price": products.iloc[original_index]["price"],
            "color": products.iloc[original_index]["color"],
            "score": similarities[position]
        })

    return results

In [92]:
results = retrieve_filtered_products(
    query="casual top for college",
    max_price=1500,
    color="Black"
)

for result in results:
    print(result)

{'product_id': 'P006', 'name': 'Black Slim Fit T-Shirt', 'price': np.int64(799), 'color': 'Black', 'score': np.float32(0.31410754)}


In [93]:
print('FAQ retrieval:working')
print('product retrieval:working')
print('metadata filtering:working')
print('filtered semantic retrieval:working')

FAQ retrieval:working
product retrieval:working
metadata filtering:working
filtered semantic retrieval:working


In [94]:
def build_product_context(results):

    context = ""

    for result in results:

        if "document" in result:

            context += f"""
{result['document']}
Similarity Score: {result['score']:.2f}

"""

        else:

            product = products[
                products["product_id"] == result["product_id"]
            ].iloc[0]

            context += f"""
Product ID: {product['product_id']}
Product: {product['name']}
Category: {product['category']}
Description: {product['description']}
Price: ₹{product['price']}
Color: {product['color']}
Available sizes: {product['size'].replace('|', ', ')}
Material: {product['material']}
Occasion: {product['occasion'].replace('|', ', ')}
Style: {product['style']}
Similarity Score: {result['score']:.2f}

"""

    return context.strip()

In [95]:
context=build_product_context(results)
print(context)

Product ID: P006
Product: Black Slim Fit T-Shirt
Category: T-Shirt
Description: Simple black slim-fit t-shirt that can be paired with jeans or trousers for everyday outfits.
Price: ₹799
Color: Black
Available sizes: S, M, L, XL
Material: Cotton
Occasion: Casual, College
Style: Minimal
Similarity Score: 0.31


In [96]:
def build_faq_context(results):

    context = ""

    for result in results:
        context += f"""
{result['document']}
Similarity Score: {result['score']:.2f}

"""

    return context.strip()

In [97]:
faq_results = retrieve_faq("Can I exchange a product?")

faq_context = build_faq_context(faq_results)

print(faq_context)

Question: Can I exchange a product?

Answer: Yes, eligible products can be exchanged for another size within 30 days of delivery.
Similarity Score: 0.77


Question: Can I return a sale item?

Answer: Sale items can be returned only if they meet the applicable return conditions.
Similarity Score: 0.48


Question: What is your return policy?

Answer: You can return eligible products within 30 days of delivery. Products must be unused and in their original condition with tags attached.
Similarity Score: 0.45


In [98]:
def build_rag_prompt(query, context):

    prompt = f"""
You are an AI shopping assistant for Fashion Forward Hub.

Use ONLY the product information provided in the context.

Answer the user's question naturally and helpfully.
Do not simply copy the context.
Do not mention similarity scores.
Do not invent product information.

If products are found, clearly mention:
- Product name
- Price
- Color
- Any other relevant information available in the context

User question:
{query}

Context:
{context}

Write a concise shopping-assistant response:
"""

    return prompt.strip()

In [99]:
query="can i exchange a product"
prompt=build_rag_prompt(query,faq_context)
print(prompt)

You are an AI shopping assistant for Fashion Forward Hub.

Use ONLY the product information provided in the context.

Answer the user's question naturally and helpfully.
Do not simply copy the context.
Do not mention similarity scores.
Do not invent product information.

If products are found, clearly mention:
- Product name
- Price
- Color
- Any other relevant information available in the context

User question:
can i exchange a product

Context:
Question: Can I exchange a product?

Answer: Yes, eligible products can be exchanged for another size within 30 days of delivery.
Similarity Score: 0.77


Question: Can I return a sale item?

Answer: Sale items can be returned only if they meet the applicable return conditions.
Similarity Score: 0.48


Question: What is your return policy?

Answer: You can return eligible products within 30 days of delivery. Products must be unused and in their original condition with tags attached.
Similarity Score: 0.45

Write a concise shopping-assistant r

In [100]:
from groq import Groq

In [101]:
import os
from dotenv import load_dotenv
load_dotenv()
client=Groq(api_key=os.getenv("GROQ_API_KEY"))
print('groq client created successfully')

groq client created successfully


In [102]:
from pathlib import Path
print(Path.cwd())

c:\Users\DELL\OneDrive\Desktop\Fashion-Forward-Hub\notebooks


In [103]:
from pathlib import Path
from dotenv import load_dotenv
import os

project_root = Path.cwd().parent
env_path = project_root / ".env"

print("Project root:", project_root)
print("File exists:", env_path.exists())

load_dotenv(env_path)

print("API key found:", os.getenv("GROQ_API_KEY") is not None)

Project root: c:\Users\DELL\OneDrive\Desktop\Fashion-Forward-Hub
File exists: True
API key found: True


In [104]:
def generate_answer(prompt):

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [105]:
answer = generate_answer(prompt)

print(answer)

Yes—eligible items can be exchanged for another size within 30 days of delivery.


In [106]:
def answer_faq(query):

    results = retrieve_faq(query, top_k=3)

    # Check whether the retrieved information is relevant enough
    if not results or results[0]["score"] < 0.60:
        return "I don't have enough information about that in the store's FAQ."

    context = build_faq_context(results)

    prompt = build_rag_prompt(query, context)

    answer = generate_answer(prompt)

    return answer

In [107]:
answer=answer_faq("can i exchange a product")
print(answer)

Yes—eligible items can be exchanged for another size within 30 days of delivery.


In [108]:
def answer_product(query):

    results = retrieve_products(query, top_k=3)

    # Check whether the retrieved products are relevant enough
    if not results or results[0]["score"] < 0.60:
        return "I couldn't find a matching product in the current catalog."

    context = build_product_context(results)

    prompt = build_rag_prompt(query, context)

    answer = generate_answer(prompt)

    return answer

In [109]:
question = "Tell me about the Black Slim Fit T-Shirt"

answer = answer_product(question)

print(answer)

**Black Slim Fit T‑Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Material:** Cotton  
- **Sizes:** S, M, L, XL  
- **Style:** Minimal, slim‑fit  
- **Occasions:** Casual, College  
- **Pairing:** Works well with jeans or trousers for everyday outfits.


In [110]:
question="i need something for college under 1500"
answer=answer_product(question)
print(answer)

I couldn't find a matching product in the current catalog.


In [111]:
results = retrieve_products("Tell me about the Black Slim Fit T-Shirt", top_k=3)

for result in results:
    index = result["index"]

    print("Product ID:", products.iloc[index]["product_id"])
    print("Product:", products.iloc[index]["name"])
    print("Similarity:", result["score"])
    print("-" * 50)

Product ID: P006
Product: Black Slim Fit T-Shirt
Similarity: 0.78567314
--------------------------------------------------
Product ID: P018
Product: Green Oversized T-Shirt
Similarity: 0.5372093
--------------------------------------------------
Product ID: P014
Product: Navy Blue Polo T-Shirt
Similarity: 0.5130693
--------------------------------------------------


In [112]:
results = retrieve_products(
    "Tell me about the Black Slim Fit T-Shirt",
    top_k=3
)

context = build_product_context(results)

print(context)

Product: Black Slim Fit T-Shirt

Category: T-Shirt

Description: Simple black slim-fit t-shirt that can be paired with jeans or trousers for everyday outfits.

Price: ₹799

Color: Black

Available sizes: S, M, L, XL

Material: Cotton

Occasion: Casual, College

Style: Minimal
Similarity Score: 0.79


Product: Green Oversized T-Shirt

Category: T-Shirt

Description: Relaxed green oversized t-shirt designed for comfortable streetwear and college outfits.

Price: ₹899

Color: Green

Available sizes: S, M, L, XL

Material: Cotton

Occasion: Casual, College

Style: Streetwear
Similarity Score: 0.54


Product: Navy Blue Polo T-Shirt

Category: Polo

Description: Classic navy polo shirt with a clean design suitable for casual and smart casual occasions.

Price: ₹1199

Color: Navy Blue

Available sizes: S, M, L, XL

Material: Cotton

Occasion: Casual, Office

Style: Smart Casual
Similarity Score: 0.51


In [113]:
prompt = build_rag_prompt(
    "Tell me about the Black Slim Fit T-Shirt",
    context
)

print(prompt)

You are an AI shopping assistant for Fashion Forward Hub.

Use ONLY the product information provided in the context.

Answer the user's question naturally and helpfully.
Do not simply copy the context.
Do not mention similarity scores.
Do not invent product information.

If products are found, clearly mention:
- Product name
- Price
- Color
- Any other relevant information available in the context

User question:
Tell me about the Black Slim Fit T-Shirt

Context:
Product: Black Slim Fit T-Shirt

Category: T-Shirt

Description: Simple black slim-fit t-shirt that can be paired with jeans or trousers for everyday outfits.

Price: ₹799

Color: Black

Available sizes: S, M, L, XL

Material: Cotton

Occasion: Casual, College

Style: Minimal
Similarity Score: 0.79


Product: Green Oversized T-Shirt

Category: T-Shirt

Description: Relaxed green oversized t-shirt designed for comfortable streetwear and college outfits.

Price: ₹899

Color: Green

Available sizes: S, M, L, XL

Material: Cotton


In [114]:
answer = generate_answer(prompt)

print(answer)

**Black Slim Fit T‑Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Material:** Cotton  
- **Sizes:** S, M, L, XL  
- **Style:** Minimal, slim‑fit  
- **Occasions:** Casual, College  
- **Pairing:** Great with jeans or trousers for everyday outfits.


In [115]:
question = "What are the available sizes and price of the Black Slim Fit T-Shirt?"

answer = answer_product(question)

print(answer)

**Black Slim Fit T-Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Available sizes:** S, M, L, XL


In [116]:
results = retrieve_filtered_products(
    query="casual top for college",
    max_price=1500,
    color="Black"
)

for result in results:
    print(result)

{'product_id': 'P006', 'name': 'Black Slim Fit T-Shirt', 'price': np.int64(799), 'color': 'Black', 'score': np.float32(0.31410754)}


In [117]:
def answer_filtered_product(query, max_price=None, color=None):

    results = retrieve_filtered_products(
        query=query,
        max_price=max_price,
        color=color,
        top_k=3
    )

    if not results:
        return "I don't have enough information to answer that."

    context = build_product_context(results)

    prompt = build_rag_prompt(query, context)

    answer = generate_answer(prompt)

    return answer

In [118]:
answer = answer_filtered_product(
    query="I want a casual black top for college",
    max_price=1500,
    color="Black"
)

print(answer)

**Black Slim Fit T‑Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Material:** Cotton  
- **Sizes:** S, M, L, XL  
- **Style:** Minimal, casual – perfect for college wear.


In [119]:
import re

def extract_max_price(query):

    query = query.lower()

    patterns = [
        r'under\s*₹?\s*(\d+)',
        r'below\s*₹?\s*(\d+)',
        r'less than\s*₹?\s*(\d+)',
        r'upto\s*₹?\s*(\d+)',
        r'up to\s*₹?\s*(\d+)'
    ]

    for pattern in patterns:
        match = re.search(pattern, query)

        if match:
            return int(match.group(1))

    return None

In [120]:
print(extract_max_price("Show me black tops under ₹1500"))
print(extract_max_price("I want something below 2000"))
print(extract_max_price("Show me a casual outfit"))

1500
2000
None


In [121]:
def extract_color(query):

    query = query.lower()

    colors = products["color"].dropna().unique()

    for color in colors:
        if color.lower() in query:
            return color

    return None

In [122]:
print(extract_color("Show me black tops under ₹1500"))
print(extract_color("I want a pink dress"))
print(extract_color("Show me something for college"))

Black
Pink
None


In [123]:
def extract_filters(query):
    max_price = extract_max_price(query)
    color = extract_color(query)
    occasion = extract_occasion(query)
    style = extract_style(query)
    category = extract_category(query)

    return {
        "max_price": max_price,
        "color": color,
        "occasion": occasion,
        "style": style,
        "category": category
    }

In [124]:
filters = extract_filters(
    "Show me black tops under ₹1500"
)

print(filters)

{'max_price': 1500, 'color': 'Black', 'occasion': None, 'style': None, 'category': 'Top'}


In [125]:
def search_products(query):
    filters = extract_filters(query)

    results = retrieve_filtered_products(
        query=query,
        max_price=filters["max_price"],
        color=filters["color"],
        occasion=filters["occasion"],
        style=filters["style"],
        category=filters["category"],
        top_k=3
    )

    return results

In [126]:
results = search_products(
    "Show me black tops under ₹1500"
)

for result in results:
    print(result)

In [127]:
for result in results:
    print(result)

In [128]:
def answer_search(query):

    results = search_products(query)

    if not results:
        return "I couldn't find any products matching your requirements."

    context = build_product_context(results)

    prompt = build_rag_prompt(query, context)

    answer = generate_answer(prompt)

    return answer

In [129]:
answer = answer_search(
    "Show me black tops under ₹1500 for college"
)

print(answer)

I couldn't find any products matching your requirements.


In [130]:
print(type(answer_search))

<class 'function'>


In [131]:
import inspect
print(inspect.getsource(answer_search))

def answer_search(query):

    results = search_products(query)

    if not results:
        return "I couldn't find any products matching your requirements."

    context = build_product_context(results)

    prompt = build_rag_prompt(query, context)

    answer = generate_answer(prompt)

    return answer



In [132]:
answer = answer_search(
    "Show me black tops under ₹1500 for college"
)

print("FINAL ANSWER:")
print(answer)

FINAL ANSWER:
I couldn't find any products matching your requirements.


In [133]:
import inspect

print(inspect.getsource(generate_answer))

def generate_answer(prompt):

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content



In [134]:
results = search_products(
    "Show me black tops under ₹1500 for college"
)

context = build_product_context(results)

prompt = build_rag_prompt(
    "Show me black tops under ₹1500 for college",
    context
)

print(prompt)

You are an AI shopping assistant for Fashion Forward Hub.

Use ONLY the product information provided in the context.

Answer the user's question naturally and helpfully.
Do not simply copy the context.
Do not mention similarity scores.
Do not invent product information.

If products are found, clearly mention:
- Product name
- Price
- Color
- Any other relevant information available in the context

User question:
Show me black tops under ₹1500 for college

Context:


Write a concise shopping-assistant response:


In [135]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

I’m sorry, but I don’t have any black tops under ₹1500 for college in my current catalog. If you’d like, I can help you look for other styles or price ranges.


In [136]:
prompt = build_rag_prompt(
    "Show me black tops under ₹1500 for college",
    context
)

answer = generate_answer(prompt)

print(answer)

I’m sorry, but I don’t have any black tops under ₹1500 for college in my current catalog. If you’d like, I can help you look for other styles or price ranges.


In [137]:
def detect_intent(query):

    query = query.lower()

    outfit_keywords = [
        "outfit",
        "look",
        "dress me",
        "style me"
    ]

    faq_keywords = [
        "return",
        "exchange",
        "shipping",
        "delivery",
        "payment",
        "cancel",
        "refund",
        "discount",
        "track order"
    ]

    search_keywords = [
        "show me",
        "find",
        "looking for",
        "under",
        "below",
        "upto",
        "up to",
        "less than"
    ]

    for keyword in outfit_keywords:
        if keyword in query:
            return "outfit"

    for keyword in search_keywords:
        if keyword in query:
            return "search"

    for keyword in faq_keywords:
        if keyword in query:
            return "faq"

    return "product"

In [138]:
print(detect_intent("What is your return policy?"))
print(detect_intent("Tell me about the black t-shirt"))
print(detect_intent("Show me black tops under ₹1500"))

faq
product
search


In [139]:
def chat(query):

    intent = detect_intent(query)

    if intent == "faq":
        return answer_faq(query)

    elif intent == "search":
        return answer_search(query)

    elif intent == "outfit":

        filters = extract_filters(query)

        return answer_outfit(
            query=query,
            occasion=filters["occasion"],
            style=filters["style"],
            max_price=filters["max_price"]
        )

    elif intent == "product":
        return answer_product(query)

In [140]:
print(chat("What is your return policy?"))

print(chat("Tell me about the Black Slim Fit T-Shirt"))

print(chat("Show me black tops under ₹1500 for college"))

Our return policy is straightforward:

- **Timeframe:** You can return eligible items within **30 days** of delivery.  
- **Condition:** Items must be **unused**, in their **original condition**, and still have the **tags attached**.  
- **Sale items:** These can be returned only if they meet the same return conditions.  

Once a return is approved and processed, refunds are issued back to the original payment method.
**Black Slim Fit T‑Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Material:** Cotton  
- **Sizes:** S, M, L, XL  
- **Style:** Minimal, slim‑fit  
- **Occasions:** Casual, College – pairs well with jeans or trousers for everyday outfits.
I couldn't find any products matching your requirements.


In [141]:
def extract_occasion(query):

    query = query.lower()

    occasions = [
        "casual",
        "college",
        "office",
        "formal",
        "summer",
        "vacation",
        "date",
        "party",
        "winter"
    ]

    for occasion in occasions:
        if occasion in query:
            return occasion.title()

    return None

In [142]:
print(extract_occasion("Show me something for college"))
print(extract_occasion("I need a casual outfit"))
print(extract_occasion("I need something for a party"))
print(extract_occasion("Show me something nice"))

College
Casual
Party
None


In [143]:
print(
    filter_products(
        max_price=1500,
        occasion="College"
    )[["product_id", "name", "price", "occasion"]]
)

   product_id                        name  price        occasion
0        P001  Classic White Cotton Shirt   1299  Casual|College
5        P006      Black Slim Fit T-Shirt    799  Casual|College
10       P011       White Casual Sneakers   1499  Casual|College
17       P018     Green Oversized T-Shirt    899  Casual|College


In [144]:
print(extract_filters("Show me black tops under ₹1500 for college"))

{'max_price': 1500, 'color': 'Black', 'occasion': 'College', 'style': None, 'category': 'Top'}


In [145]:
results = search_products(
    "Show me black tops under ₹1500 for college"
)

print(results)

[]


In [146]:
print(answer_search("Show me black tops under ₹1500 for college"))

I couldn't find any products matching your requirements.


In [147]:
results = search_products(
    "Show me black tops under ₹1500 for college"
)

context = build_product_context(results)

print(context)

In [148]:
query = "Show me black tops under ₹1500 for college"

prompt = build_rag_prompt(
    query,
    context
)

answer = generate_answer(prompt)

print(answer)

I’m sorry, but I don’t have any black tops under ₹1500 for college in my current catalog. If you’d like, I can help you look for other styles or price ranges.


In [149]:
results = search_products(
    "Show me a black dress under ₹500"
)

print(results)

[]


In [150]:
print(chat("Show me a black dress under ₹500"))

I couldn't find any products matching your requirements.


In [151]:
print(chat("How long does shipping take?"))

print(chat("What sizes are available for the Black Slim Fit T-Shirt?"))

print(chat("Show me casual clothes under ₹1500"))

print(chat("Show me pink products for college"))

print(chat("Do you offer free shipping?"))

Shipping typically takes 3 to 5 business days for standard delivery. Times can vary slightly depending on your location.
**Black Slim Fit T-Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Available sizes:** S, M, L, XL
Here are some casual items priced under ₹1500:

| Product | Price | Color | Key Details |
|---------|-------|-------|-------------|
| Black Slim Fit T‑Shirt | ₹799 | Black | Cotton, slim‑fit, great for jeans or trousers |
| White Casual Sneakers | ₹1499 | White | Synthetic leather, minimal style, fits sizes 6‑10 |
| Lavender Casual Skirt | ₹1499 | Lavender | Cotton‑blend, feminine style, pairs well with tops |

All of these are casual‑wear options within your budget. Let me know if you’d like more details or additional styles!
I couldn't find any products matching your requirements.
Yes—free standard shipping is available on orders over ₹1999.


In [152]:
def extract_style(query):

    query = query.lower()

    styles = products["style"].dropna().unique()

    for style in styles:
        if style.lower() in query:
            return style

    return None

In [153]:
print(extract_style("Show me something minimal"))
print(extract_style("I want a slim fit outfit"))
print(extract_style("Show me something nice"))

Minimal
None
None


In [154]:
print(extract_style("Show me something oversized"))
print(extract_style("I want a minimal style"))
print(extract_style("Show me cargo style"))

None
Minimal
None


In [155]:
print(products["style"].unique())

['Minimal' 'Streetwear' 'Classic' 'Smart Casual' 'Feminine' 'Formal'
 'Partywear' 'Modern']


In [156]:
print(extract_style("Show me a minimal outfit"))
print(extract_style("I want something streetwear"))
print(extract_style("Show me a formal outfit"))
print(extract_style("I want a partywear dress"))

Minimal
Streetwear
Formal
Partywear


In [157]:
print(
    filter_products(style="Formal")
    [["product_id", "name", "price", "style"]]
)

   product_id                         name  price   style
8        P009  Black Straight Fit Trousers   1799  Formal
18       P019                 Black Blazer   2999  Formal


In [158]:
print(extract_filters(
    "Show me formal black clothes under ₹2500"
))

{'max_price': 2500, 'color': 'Black', 'occasion': 'Formal', 'style': 'Formal', 'category': None}


In [159]:
print(products["occasion"].unique())

['Casual|College' 'Casual|Office' 'Summer|Vacation' 'Office|Formal'
 'Casual|Date' 'Party|Date' 'Winter|Casual' 'Formal|Office']


In [160]:
print(extract_occasion("Show me something for college"))
print(extract_occasion("I need a formal outfit"))
print(extract_occasion("I need something for a party"))
print(extract_occasion("Show me summer clothes"))

College
Formal
Party
Summer


In [161]:
print(extract_filters(
    "Show me formal black clothes under ₹2500"
))

{'max_price': 2500, 'color': 'Black', 'occasion': 'Formal', 'style': 'Formal', 'category': None}


In [162]:
results = search_products(
    "Show me formal black clothes under ₹2500"
)

print(results)

[{'product_id': 'P009', 'name': 'Black Straight Fit Trousers', 'price': np.int64(1799), 'color': 'Black', 'score': np.float32(0.6377317)}]


In [163]:
print(chat("Show me formal black clothes under ₹2500"))

**Formal Black Outfit Under ₹2500**

- **Product:** Black Straight Fit Trousers  
- **Price:** ₹1799  
- **Color:** Black  
- **Style:** Formal (office, smart casual)  
- **Material:** Polyester Blend  
- **Sizes Available:** S, M, L, XL  

This pair of trousers is a great, budget‑friendly option for formal occasions.


In [164]:
print(products["category"].unique())

['Shirt' 'Hoodie' 'Jeans' 'Trousers' 'Dress' 'T-Shirt' 'Jacket' 'Top'
 'Shoes' 'Pants' 'Polo' 'Sweater' 'Blazer' 'Skirt']


In [165]:
def extract_category(query):
    query = query.lower()

    category_map = {
        "shirt": "Shirt",
        "shirts": "Shirt",
        "hoodie": "Hoodie",
        "hoodies": "Hoodie",
        "jeans": "Jeans",
        "trouser": "Trousers",
        "trousers": "Trousers",
        "dress": "Dress",
        "dresses": "Dress",
        "t-shirt": "T-Shirt",
        "t-shirts": "T-Shirt",
        "jacket": "Jacket",
        "jackets": "Jacket",
        "top": "Top",
        "tops": "Top",
        "shoe": "Shoes",
        "shoes": "Shoes",
        "pant": "Pants",
        "pants": "Pants",
        "polo": "Polo",
        "sweater": "Sweater",
        "sweaters": "Sweater",
        "blazer": "Blazer",
        "blazers": "Blazer",
        "skirt": "Skirt",
        "skirts": "Skirt"
    }

    for word, category in category_map.items():
        if word in query:
            return category

    return None

In [166]:
print(extract_category("show me black shirts"))
print(extract_category("find me a dress"))
print(extract_category("I need a hoodie"))
print(extract_category("show me shoes under ₹1500"))
print(extract_category("show me something nice"))

Shirt
Dress
Hoodie
Shoes
None


In [167]:
print(extract_filters("Show me black shirts for college under ₹1500"))

{'max_price': 1500, 'color': 'Black', 'occasion': 'College', 'style': None, 'category': 'Shirt'}


In [168]:
print(filter_products(category="Shirt")[
    ["product_id", "name", "category", "price"]
])

  product_id                        name category  price
0       P001  Classic White Cotton Shirt    Shirt   1299
7       P008           Beige Linen Shirt    Shirt   1599


In [169]:
print(
    retrieve_filtered_products(
        query="black shirts for college",
        category="Shirt"
    )
)

[{'product_id': 'P001', 'name': 'Classic White Cotton Shirt', 'price': np.int64(1299), 'color': 'White', 'score': np.float32(0.49694127)}, {'product_id': 'P008', 'name': 'Beige Linen Shirt', 'price': np.int64(1599), 'color': 'Beige', 'score': np.float32(0.4662363)}]


In [170]:
print(search_products("Show me black shirts under ₹1500"))

[]


In [171]:
print(search_products("Show me white shirts under ₹1500"))

[{'product_id': 'P001', 'name': 'Classic White Cotton Shirt', 'price': np.int64(1299), 'color': 'White', 'score': np.float32(0.5640759)}]


In [172]:
print(extract_category("show me shirts"))
print(extract_category("I want dresses"))
print(extract_category("find some shoes"))
print(extract_category("show me jeans"))
print(extract_category("something nice"))

Shirt
Dress
Shoes
Jeans
None


In [173]:
print(chat("Show me white shirts under ₹1500"))

**Classic White Cotton Shirt**  
- **Price:** ₹1299  
- **Color:** White  
- **Material:** Cotton  
- **Fit:** Regular  
- **Occasion:** Casual, College  
- **Sizes Available:** S, M, L, XL  

This shirt is under ₹1500 and fits your criteria.


In [174]:
print(
    products[
        ['product_id','name','category','price','occasion','style']
    ].to_string(index=False)
    
)

product_id                        name category  price        occasion        style
      P001  Classic White Cotton Shirt    Shirt   1299  Casual|College      Minimal
      P002      Black Oversized Hoodie   Hoodie   1799  Casual|College   Streetwear
      P003     Blue Straight Fit Jeans    Jeans   1899  Casual|College      Classic
      P004     Beige Wide Leg Trousers Trousers   1699   Casual|Office Smart Casual
      P005         Floral Summer Dress    Dress   1999 Summer|Vacation     Feminine
      P006      Black Slim Fit T-Shirt  T-Shirt    799  Casual|College      Minimal
      P007     Light Blue Denim Jacket   Jacket   2199  Casual|College   Streetwear
      P008           Beige Linen Shirt    Shirt   1599 Summer|Vacation      Minimal
      P009 Black Straight Fit Trousers Trousers   1799   Office|Formal       Formal
      P010             Pink Ribbed Top      Top    999     Casual|Date     Feminine
      P011       White Casual Sneakers    Shoes   1499  Casual|College      

In [175]:
def retrieve_outfit_products(
    occasion=None,
    style=None,
    max_price=None
):

    filtered = products.copy()

    # 1. Occasion is a hard requirement
    if occasion is not None:
        filtered = filtered[
            filtered["occasion"].str.lower().str.contains(
                occasion.lower(),
                na=False
            )
        ]

    # 2. Budget is a hard requirement
    if max_price is not None:
        filtered = filtered[
            filtered["price"] <= max_price
        ]

    if filtered.empty:
        return filtered

    # 3. Style is a preference
    if style is not None:
        filtered["style_match"] = (
            filtered["style"].str.lower() == style.lower()
        )

        filtered = filtered.sort_values(
            "style_match",
            ascending=False
        )

    # 4. Select outfit categories

    selected = []
    # Dresses can be complete outfits by themselves

    dresses = filtered[
        filtered["category"] == "Dress"
]

    if not dresses.empty:
        selected.append(dresses.iloc[0])

        return pd.DataFrame(selected)
    tops = filtered[
        filtered["category"].isin(
        ["Shirt", "T-Shirt", "Hoodie", "Polo", "Top"]
    )
]

    bottoms = filtered[
        filtered["category"].isin(
        ["Jeans", "Pants", "Trousers", "Skirt"]
    )
]

    shoes = filtered[
    filtered["category"] == "Shoes"
]

    layers = filtered[
        filtered["category"].isin(
        ["Jacket", "Blazer"]
    )
]

    # Find an affordable top + bottom combination
    if not tops.empty and not bottoms.empty:

        found_combination = False

    for _, top in tops.iterrows():

        for _, bottom in bottoms.iterrows():

            total = top["price"] + bottom["price"]

            if max_price is None or total <= max_price:

                selected = [top, bottom]
                found_combination = True
                break

        if found_combination:
            break

    # Add shoes if they fit within the budget
    if selected and not shoes.empty:

        current_total = sum(product["price"] for product in selected)

        for _, shoe in shoes.iterrows():

            if (
                max_price is None
                or current_total + shoe["price"] <= max_price
        ):
                selected.append(shoe)
                break

    # Add an optional layer if it fits
    if selected and not layers.empty:

        current_total = sum(product["price"] for product in selected)

        for _, layer in layers.iterrows():

            if (
                max_price is None
                or current_total + layer["price"] <= max_price
        ):
                selected.append(layer)
                break

    result = pd.DataFrame(selected)

    if "style_match" in result.columns:
        result = result.drop(columns=["style_match"])

    return result

In [176]:
print(
    retrieve_outfit_products(
        occasion="College",
        style="Minimal"
    )[["product_id", "name", "category", "price", "style"]]
)

   product_id                        name category  price       style
0        P001  Classic White Cotton Shirt    Shirt   1299     Minimal
2        P003     Blue Straight Fit Jeans    Jeans   1899     Classic
10       P011       White Casual Sneakers    Shoes   1499     Minimal
6        P007     Light Blue Denim Jacket   Jacket   2199  Streetwear


In [177]:
def build_outfit_prompt(query, products_context):

    prompt = f"""
You are an AI fashion shopping assistant for Fashion Forward Hub.

The user wants an outfit recommendation.

Use ONLY the products provided in the context.
Do not invent products, prices, colors, sizes, or other details.

Create a practical outfit by combining compatible products from the available options.

For each recommended product, mention:
- Product name
- Price
- Color

Also give a short explanation of why the combination works for the requested occasion or style.

User request:
{query}

Available products:
{products_context}

Give a concise and natural outfit recommendation.
"""

    return prompt.strip()

In [178]:
def build_outfit_context(outfit_products):

    context = ""

    for _, product in outfit_products.iterrows():

        context += f"""
Product ID: {product['product_id']}
Product: {product['name']}
Category: {product['category']}
Price: ₹{product['price']}
Color: {product['color']}
Material: {product['material']}
Occasion: {product['occasion'].replace('|', ', ')}
Style: {product['style']}

"""

    return context.strip()

In [179]:
outfit_products = retrieve_outfit_products(
    occasion="College",
    style="Minimal"
)

outfit_context = build_outfit_context(outfit_products)

print(outfit_context)

Product ID: P001
Product: Classic White Cotton Shirt
Category: Shirt
Price: ₹1299
Color: White
Material: Cotton
Occasion: Casual, College
Style: Minimal


Product ID: P003
Product: Blue Straight Fit Jeans
Category: Jeans
Price: ₹1899
Color: Blue
Material: Cotton Denim
Occasion: Casual, College
Style: Classic


Product ID: P011
Product: White Casual Sneakers
Category: Shoes
Price: ₹1499
Color: White
Material: Synthetic Leather
Occasion: Casual, College
Style: Minimal


Product ID: P007
Product: Light Blue Denim Jacket
Category: Jacket
Price: ₹2199
Color: Light Blue
Material: Denim
Occasion: Casual, College
Style: Streetwear


In [180]:
def answer_outfit(query, occasion=None, style=None, max_price=None):

    outfit_products = retrieve_outfit_products(
        occasion=occasion,
        style=style,
        max_price=max_price
    )

    if outfit_products.empty:
        return "I couldn't find enough products for this outfit."

    context = build_outfit_context(outfit_products)

    prompt = build_outfit_prompt(
        query,
        context
    )

    answer = generate_answer(prompt)

    return answer

In [181]:
print(
    answer_outfit(
        "Suggest a minimal outfit for college",
        occasion="College",
        style="Minimal"
    )
)

**Minimal College Outfit**

| Product | Price | Color |
|---------|-------|-------|
| Classic White Cotton Shirt | ₹1299 | White |
| Blue Straight Fit Jeans | ₹1899 | Blue |
| White Casual Sneakers | ₹1499 | White |

**Why it works:**  
The crisp white shirt and relaxed blue jeans create a clean, timeless look that’s easy to layer and comfortable for a day of classes. Pairing them with white sneakers keeps the outfit minimal yet stylish, perfect for a casual college vibe.


In [182]:
print(detect_intent("Suggest an outfit for college"))
print(detect_intent("Show me black shirts"))
print(detect_intent("What is your return policy?"))
print(detect_intent("Tell me about the black t-shirt"))

outfit
search
faq
product


In [183]:
print(chat("Suggest a minimal outfit for college"))

**Minimal College Outfit**

| Product | Price | Color |
|---------|-------|-------|
| Classic White Cotton Shirt | ₹1299 | White |
| Blue Straight Fit Jeans | ₹1899 | Blue |
| White Casual Sneakers | ₹1499 | White |

**Why it works:**  
The crisp white shirt and clean‑cut blue jeans create a timeless, low‑maintenance look that’s perfect for campus life. Pairing them with white sneakers keeps the outfit fresh and comfortable, while the monochrome palette ensures easy styling and a polished minimal aesthetic.


In [184]:
print(
    products[
        products["occasion"].str.lower().str.contains("college", na=False)
    ][["product_id", "name", "category", "style"]].to_string(index=False)
)

product_id                       name category      style
      P001 Classic White Cotton Shirt    Shirt    Minimal
      P002     Black Oversized Hoodie   Hoodie Streetwear
      P003    Blue Straight Fit Jeans    Jeans    Classic
      P006     Black Slim Fit T-Shirt  T-Shirt    Minimal
      P007    Light Blue Denim Jacket   Jacket Streetwear
      P011      White Casual Sneakers    Shoes    Minimal
      P013          Olive Cargo Pants    Pants Streetwear
      P018    Green Oversized T-Shirt  T-Shirt Streetwear


In [185]:
print(
    retrieve_outfit_products(
        occasion="College",
        style="Minimal"
    )[["product_id", "name", "category", "price"]]
)

   product_id                        name category  price
0        P001  Classic White Cotton Shirt    Shirt   1299
2        P003     Blue Straight Fit Jeans    Jeans   1899
10       P011       White Casual Sneakers    Shoes   1499
6        P007     Light Blue Denim Jacket   Jacket   2199


In [186]:
print(
    retrieve_outfit_products(
        occasion="College",
        style="Minimal"
    )[["product_id", "name", "category", "price", "style"]]
)

   product_id                        name category  price       style
0        P001  Classic White Cotton Shirt    Shirt   1299     Minimal
2        P003     Blue Straight Fit Jeans    Jeans   1899     Classic
10       P011       White Casual Sneakers    Shoes   1499     Minimal
6        P007     Light Blue Denim Jacket   Jacket   2199  Streetwear


In [187]:
print(
    retrieve_outfit_products(
        occasion="College",
        style="Minimal"
    )[["product_id", "name", "category", "price", "style"]]
)

   product_id                        name category  price       style
0        P001  Classic White Cotton Shirt    Shirt   1299     Minimal
2        P003     Blue Straight Fit Jeans    Jeans   1899     Classic
10       P011       White Casual Sneakers    Shoes   1499     Minimal
6        P007     Light Blue Denim Jacket   Jacket   2199  Streetwear


In [188]:
print(
    chat("Suggest a minimal outfit for college")
)

**Minimal College Outfit**

| Product | Price | Color |
|---------|-------|-------|
| Classic White Cotton Shirt | ₹1299 | White |
| Blue Straight Fit Jeans | ₹1899 | Blue |
| White Casual Sneakers | ₹1499 | White |

**Why it works:**  
The crisp white shirt and clean‑cut blue jeans create a timeless, low‑maintenance look that’s perfect for campus life. Pairing them with white sneakers keeps the outfit fresh and comfortable, while the monochrome palette ensures easy styling and a polished minimal aesthetic.


In [189]:
print(
    chat("Suggest a minimal college outfit under ₹4000")
)

**Minimal College Outfit (₹3,198)**  

| Product | Price | Color |
|---------|-------|-------|
| Classic White Cotton Shirt | ₹1,299 | White |
| Blue Straight Fit Jeans | ₹1,899 | Blue |

**Why it works:**  
The crisp white shirt pairs perfectly with the classic blue jeans for a clean, understated look that’s both comfortable and stylish—ideal for a college day. The combination stays well under ₹4,000 while keeping the minimal aesthetic you’re after.


In [190]:
outfit = retrieve_outfit_products(
    occasion="College",
    style="Minimal",
    max_price=4000
)

print(outfit[["product_id", "name", "category", "price"]])
print("Total:", outfit["price"].sum())

  product_id                        name category  price
0       P001  Classic White Cotton Shirt    Shirt   1299
2       P003     Blue Straight Fit Jeans    Jeans   1899
Total: 3198


In [191]:
print(
    chat("Suggest a minimal college outfit under ₹2000")
)

I couldn't find enough products for this outfit.


In [192]:
print(
    retrieve_outfit_products(
        occasion="College",
        max_price=3000
    )[["product_id", "name", "category", "price"]]
)

print("Total:", retrieve_outfit_products(
    occasion="College",
    max_price=3000
)["price"].sum())

  product_id                     name category  price
5       P006   Black Slim Fit T-Shirt  T-Shirt    799
2       P003  Blue Straight Fit Jeans    Jeans   1899
Total: 2698


In [193]:
outfit = retrieve_outfit_products(
    occasion="College",
    max_price=3000
)

print(outfit[["product_id", "name", "category", "price"]])
print("Total:", outfit["price"].sum())

  product_id                     name category  price
5       P006   Black Slim Fit T-Shirt  T-Shirt    799
2       P003  Blue Straight Fit Jeans    Jeans   1899
Total: 2698


In [194]:
print(
    chat("Suggest a minimal college outfit under ₹3000")
)

**Minimal College Outfit (₹3000 budget)**  

| Product | Price | Color |
|---------|-------|-------|
| Black Slim Fit T‑Shirt | ₹799 | Black |
| Blue Straight Fit Jeans | ₹1899 | Blue |

**Why it works:**  
The black tee offers a clean, versatile base that’s perfect for a minimal look, while the blue straight‑fit jeans add a classic, relaxed vibe. Together they create a simple yet polished outfit that’s comfortable for campus life and stays well under the ₹3000 limit.


In [195]:
print(
    chat("Suggest a party outfit under ₹1000")
)

I couldn't find enough products for this outfit.


In [196]:
print(
    chat("Suggest a party outfit")
)

**Party Outfit Recommendation**

- **Product:** Red Party Dress  
- **Price:** ₹2499  
- **Color:** Red  

This striking red dress is perfect for a party setting. Its vibrant color and party‑wear style make it a standout choice, while the polyester material ensures comfort and a polished look throughout the evening. Pair it with your favorite accessories to complete the look.


In [197]:
print(
    chat("Suggest a streetwear outfit for college")
)

**Streetwear college look**

| Product | Price | Color |
|---------|-------|-------|
| Black Oversized Hoodie | ₹1799 | Black |
| Olive Cargo Pants | ₹1799 | Olive |
| White Casual Sneakers | ₹1499 | White |
| Light Blue Denim Jacket | ₹2199 | Light Blue |

**Why it works:**  
The black hoodie gives a relaxed, urban vibe, while the olive cargo pants add functional style with plenty of pockets. White sneakers keep the look fresh and versatile for campus walks. Layer the light‑blue denim jacket for extra edge and warmth—perfect for those cooler college days. Together, they form a cohesive, street‑ready outfit that’s comfortable, practical, and on‑trend.


In [198]:
print(
    chat("Tell me about the Beige Linen Shirt")
)

**Beige Linen Shirt**  
- **Price:** ₹1599  
- **Color:** Beige  
- **Material:** Linen  
- **Sizes:** S, M, L, XL  
- **Occasion:** Ideal for summer and vacation wear  
- **Style:** Minimal, breathable design perfect for warm weather and relaxed outfits.


In [199]:
print(
    chat("How long does shipping take?")
)

Shipping typically takes 3 to 5 business days for standard delivery. Times can vary slightly depending on your location.


In [200]:
print(
    chat("Do you offer cash on delivery?")
)

I don't have enough information about that in the store's FAQ.


In [201]:
print(
    chat("Tell me about the red leather jacket")
)

I couldn't find a matching product in the current catalog.


In [202]:
print(
    chat("Tell me about the Black Slim Fit T-Shirt")
)

**Black Slim Fit T‑Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Material:** Cotton  
- **Sizes:** S, M, L, XL  
- **Style:** Minimal, slim‑fit  
- **Occasions:** Casual, College  
- **Pairing:** Works well with jeans or trousers for everyday outfits.


In [203]:
print(
    chat("Show me white shirts under ₹1500")
)

**Classic White Cotton Shirt**  
- **Price:** ₹1299  
- **Color:** White  
- **Material:** Cotton  
- **Fit:** Regular  
- **Occasion:** Casual, College  
- **Sizes Available:** S, M, L, XL  

This shirt is under ₹1500 and fits your criteria.


In [204]:
import inspect

print(inspect.getsource(retrieve_filtered_products))

def retrieve_filtered_products(
    query,
    max_price=None,
    color=None,
    occasion=None,
    style=None,
    category=None,
    top_k=3
):

    filtered = filter_products(
        max_price=max_price,
        color=color,
        occasion=occasion,
        style=style,
        category=category
    )

    if filtered.empty:
        return []

    # Get the original dataframe indices
    filtered_indices = filtered.index.tolist()

    # Create embedding for the user's query
    query_embedding = model.encode(query)

    # Compare query with only the filtered products
    similarities = cosine_similarity(
        [query_embedding],
        product_embeddings[filtered_indices]
    )[0]

    # Get the most relevant products
    top_positions = similarities.argsort()[::-1][:top_k]

    results = []

    for position in top_positions:

        original_index = filtered_indices[position]

        results.append({
            "product_id": products.iloc[original_index]["product_id"],


In [205]:
filters=extract_filters(
    "Show me black tops under ₹1500"
)
print(filters)

{'max_price': 1500, 'color': 'Black', 'occasion': None, 'style': None, 'category': 'Top'}


In [207]:
filtered = filter_products(
    max_price=1500,
    color="Black",
    category="Top"
)

print(
    filtered[
        ["product_id", "name", "category", "price", "color"]
    ]
)

Empty DataFrame
Columns: [product_id, name, category, price, color]
Index: []


In [208]:
filtered = filter_products(
    max_price=1500,
    color="Pink",
    category="Top"
)

print(
    filtered[
        ["product_id", "name", "category", "price", "color"]
    ]
)

  product_id             name category  price color
9       P010  Pink Ribbed Top      Top    999  Pink


In [209]:
results = search_products(
    "Show me pink tops under ₹1500"
)

print(results)

[{'product_id': 'P010', 'name': 'Pink Ribbed Top', 'price': np.int64(999), 'color': 'Pink', 'score': np.float32(0.55546325)}]


In [210]:
print(
    chat("Show me pink tops under ₹1500")
)

**Pink Ribbed Top**  
- **Price:** ₹999  
- **Color:** Pink  
- **Material:** Cotton Blend  
- **Sizes:** S, M, L, XL  
- **Occasion:** Casual, Date  
- **Style:** Feminine  

This top fits your criteria of being pink and under ₹1500.


In [211]:
test_queries = [
    "How long does shipping take?",
    "Tell me about the Black Slim Fit T-Shirt",
    "Show me white shirts under ₹1500",
    "Do you offer cash on delivery?",
    "Tell me about the red leather jacket",
    "Suggest a minimal college outfit under ₹3000",
    "Suggest a party outfit",
    "Suggest a streetwear outfit for college"
]

for query in test_queries:

    print("\n" + "=" * 70)
    print("USER:", query)
    print("-" * 70)
    print("ASSISTANT:", chat(query))


USER: How long does shipping take?
----------------------------------------------------------------------
ASSISTANT: Shipping typically takes 3 to 5 business days for standard delivery. Times can vary slightly depending on your location.

USER: Tell me about the Black Slim Fit T-Shirt
----------------------------------------------------------------------
ASSISTANT: **Black Slim Fit T‑Shirt**  
- **Price:** ₹799  
- **Color:** Black  
- **Material:** Cotton  
- **Sizes:** S, M, L, XL  
- **Style:** Minimal, slim‑fit  
- **Occasions:** Casual, College  
- **Pairing:** Works well with jeans or trousers for everyday outfits.

USER: Show me white shirts under ₹1500
----------------------------------------------------------------------
ASSISTANT: **Classic White Cotton Shirt**  
- **Price:** ₹1299  
- **Color:** White  
- **Material:** Cotton  
- **Fit:** Regular  
- **Occasion:** Casual, College  
- **Sizes Available:** S, M, L, XL  

This shirt is under ₹1500 and fits your criteria.

USER